In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
from keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


## ResNet

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, (224, 224))
    image = tf.keras.applications.resnet50.preprocess_input(image)
    return image, label

train_res = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_res = train_res.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

test_res = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_res = test_res.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

IMG_SIZE = 224
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_res, epochs=5, validation_data=test_res)

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 189s 114ms/step - accuracy: 0.9026 - loss: 0.2887 - val_accuracy: 0.8970 - val_loss: 0.2972
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 177s 100ms/step - accuracy: 0.9233 - loss: 0.2182 - val_accuracy: 0.9013 - val_loss: 0.2914
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 170s 109ms/step - accuracy: 0.9383 - loss: 0.1755 - val_accuracy: 0.9025 - val_loss: 0.2966
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 187s 99ms/step - accuracy: 0.9498 - loss: 0.1430 - val_accuracy: 0.9024 - val_loss: 0.3161
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 202s 99ms/step - accuracy: 0.9572 - loss: 0.1189 - val_accuracy: 0.8989 - val_loss: 0.3546


In [ ]:
base_model.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_res, epochs=2, validation_data=test_res)

Epoch 1/2
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 555s 321ms/step - accuracy: 0.9052 - loss: 0.3109 - val_accuracy: 0.9238 - val_loss: 0.2380
Epoch 2/2
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 480s 307ms/step - accuracy: 0.9825 - loss: 0.0590 - val_accuracy: 0.9345 - val_loss: 0.2340


## Inception

In [13]:
IMG_SIZE = 299
BATCH_SIZE = 32
NUM_CLASSES = 10
AUTOTUNE = tf.data.AUTOTUNE

# Preprocessing function for InceptionV3
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.inception_v3.preprocess_input(image)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras.applications import InceptionV3

base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # freeze the base

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [11]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_ds, epochs=5, validation_data=val_ds)

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 252s 148ms/step - accuracy: 0.8069 - loss: 0.5832 - val_accuracy: 0.8642 - val_loss: 0.3984
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 242s 142ms/step - accuracy: 0.8746 - loss: 0.3654 - val_accuracy: 0.8578 - val_loss: 0.4323
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 261s 142ms/step - accuracy: 0.8861 - loss: 0.3280 - val_accuracy: 0.8647 - val_loss: 0.4087
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 254s 137ms/step - accuracy: 0.8972 - loss: 0.2946 - val_accuracy: 0.8733 - val_loss: 0.3832
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 225s 143ms/step - accuracy: 0.9077 - loss: 0.2647 - val_accuracy: 0.8755 - val_loss: 0.3817


In [1]:
base_model.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_ds, epochs=2, validation_data=val_ds)

NameError: name 'base_model' is not defined